In [ ]:
# Cell 1
!pip install --upgrade google-cloud-aiplatform
!pip install mcp

In [ ]:
# Cell 2
import vertexai
from vertexai.preview import reasoning_engines
from google.adk.agents.callback_context import CallbackContext
from google.adk.agents import Agent
from google.adk.models import LlmRequest, LlmResponse

from typing import Optional

In [ ]:
# Cell 3
import vertexai
from vertexai.generative_models import GenerativeModel

staging_bucket = 'gs://fema_case_study_bucket'

vertexai.init(project='qwiklabs-gcp-00-117e2d1e6738', location='global', staging_bucket=staging_bucket)

In [ ]:
# Cell 4
import requests
from typing import Optional, Dict, List

def get_extended_weather_forecast(lat: float, lon: float) -> Optional[Dict]:
  """
  Retrieves weather forecast data from the U.S. National Weather Service API
  using latitude and longitude to first find the forecast endpoint.

  Args:
      lat (float): The latitude of the location (e.g., 36.9741).
      lon (float): The longitude of the location (e.g., -122.0308).

  Returns:
      Optional[Dict]: A dictionary containing the weather forecast data,
                      or None if an error occurs or data is not available.
  """
  # Step 1: Construct the URL for the /points endpoint
  points_url = f"https://api.weather.gov/points/{lat},{lon}"

  # Step 2: Make the request to the /points endpoint
  # It's good practice to include a User-Agent header for NWS API requests.
  headers = {'User-Agent': 'Google Colab Weather Agent (stephen.zott@afs.com)'}
  try:
    points_response = requests.get(points_url, headers=headers)
    points_response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
    points_data = points_response.json()
  except requests.exceptions.RequestException as e:
    print(f"Error fetching points data from NWS API: {e}")
    return None

  # Step 3: Extract the forecast URL from the response
  forecast_url = points_data.get('properties', {}).get('forecast')
  if not forecast_url:
    print("Could not find forecast URL in NWS points response.")
    return None

  # Step 4: Make the request to the forecast URL
  try:
    forecast_response = requests.get(forecast_url, headers=headers)
    forecast_response.raise_for_status()
    forecast_data = forecast_response.json()
    return forecast_data
  except requests.exceptions.RequestException as e:
    print(f"Error fetching forecast data from NWS API: {e}")
    return None



In [ ]:
# Cell 5
import getpass

# Securely prompt for the API key
MAPS_API_KEY = getpass.getpass('Enter your Google Maps API key: ')

In [ ]:
# Cell 6
from typing import Optional, Tuple
def get_lat_long(location: str, api_key: str) -> Optional[Tuple[float, float]]:
  """
  Converts a location string into latitude and longitude.
  """
  base_url = "https://maps.googleapis.com/maps/api/geocode/json"
  params = {
      "address": location,
      "key": api_key
  }

  try:
    response = requests.get(base_url, params=params)
    response.raise_for_status()
    data = response.json()
  except requests.exceptions.RequestException as e:
    print(f"Error fetching geocoding data: {e}")
    return None

  if data.get("status") == "OK" and data.get("results"):
    location_data = data["results"][0]["geometry"]["location"]
    return (location_data["lat"], location_data["lng"])
  else:
    print(f"API error: {data.get('status')}")
    return None

In [ ]:
# Cell 7
from google.adk.agents import Agent
import os

# Ensure the key is in the environment for the tools to access
os.environ['MAPS_API_KEY'] = MAPS_API_KEY

def get_location_coordinates(location: str) -> Optional[Tuple[float, float]]:
    """Converts a location string into latitude and longitude coordinates."""
    # Access the key from environment variables
    api_key = os.environ.get('MAPS_API_KEY')
    return get_lat_long(location, api_key)

In [ ]:
# Cell 8
def get_directions(origin: str, destination: str, mode: str = "driving") -> Optional[Dict]:
  """
  Retrieves directions between two locations using the Google Maps Directions API.

  Args:
      origin (str): The starting address or location.
      destination (str): The destination address or location.
      mode (str): Travel mode - driving, walking, bicycling, or transit.

  Returns:
      Optional[Dict]: A dictionary with a list of alternative routes, each
                      containing distance, duration, and step-by-step
                      instructions, or None if an error occurs.
  """
  base_url = "https://maps.googleapis.com/maps/api/directions/json"
  api_key = os.environ.get('MAPS_API_KEY')
  params = {
      "origin": origin,
      "destination": destination,
      "mode": mode,
      "alternatives": "true",
      "key": api_key,
  }

  try:
    response = requests.get(base_url, params=params)
    response.raise_for_status()
    data = response.json()
  except requests.exceptions.RequestException as e:
    print(f"Error fetching directions data: {e}")
    return None

  if data.get("status") != "OK" or not data.get("routes"):
    print(f"API error: {data.get('status')}")
    return None

  routes = []
  for route in data["routes"]:
    leg = route["legs"][0]
    steps = [
        step["html_instructions"]
        for step in leg.get("steps", [])
    ]
    routes.append({
        "summary": route.get("summary", ""),
        "origin": leg["start_address"],
        "destination": leg["end_address"],
        "distance": leg["distance"]["text"],
        "duration": leg["duration"]["text"],
        "steps": steps,
    })

  return {"routes": routes}

In [ ]:
# Cell 9
import logging
import sys

def setup_callback_logger(name="callback_logger", level=logging.INFO):
    """Configures and returns a logger for use in callback loops."""
    logger = logging.getLogger(name)
    logger.setLevel(level)

    # Clear existing handlers to avoid duplicate logs in Colab
    if logger.hasHandlers():
        logger.handlers.clear()

    handler = logging.StreamHandler(sys.stdout)
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    handler.setFormatter(formatter)
    logger.addHandler(handler)
    return logger

# Initialize the logger
callback_logger = setup_callback_logger()

In [ ]:
# Cell 10
def moderate_user_prompt(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:
    """Checks if the user prompt is valid and provides specific feedback for failures."""
    logger = logging.getLogger("callback_logger")
    try:
        if not llm_request.contents:
            return None

        last = llm_request.contents[-1]
        if not last.parts or not last.parts[0].text:
            return None

        user_text = last.parts[0].text.strip()
        result = check_user_input(user_text)

        if result == "WEATHER_OUTSIDE":
            return LlmResponse(content={
                "role": "model",
                "parts": [{"text": "I'm sorry, I can only provide specific weather reports for locations within the United States. However, for other info, I can try searching the web!"}]
            })
        elif result == "BAD":
            return LlmResponse(content={
                "role": "model",
                "parts": [{"text": "I'm sorry, I cannot fulfill this request as it violates safety guidelines."}]
            })
    except Exception as e:
        logger.exception(f"Moderation callback failed: {e}")

    return None

In [ ]:
# Cell 11
def log_user_prompt(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:
    """Logs which agent is currently handling the request and the user input."""
    logger = logging.getLogger("callback_logger")
    if llm_request.contents:
        last = llm_request.contents[-1]
        if last.role == "user" and last.parts and last.parts[0].text:
            # Log the agent name from the context to see delegation
            logger.info(f"[DELEGATION] Request being handled by Agent: {callback_context.agent_name}")
            logger.info("[%s] USER >> %s", callback_context.agent_name, last.parts[0].text.strip())
    return None

In [ ]:
# Cell 12
def log_model_response(callback_context: CallbackContext, llm_response: LlmResponse) -> Optional[LlmResponse]:
    """Logs the model response to the callback logger."""
    logger = logging.getLogger("callback_logger")
    if llm_response.content and llm_response.content.parts:
        txt = llm_response.content.parts[0].text
        if txt:
            logger.info("[%s] MODEL >> %s", callback_context.agent_name, txt.strip())

    return None

In [ ]:
# Cell 13
def check_user_input(text: str) -> str:
    """
    Analyzes user input.
    Returns 'BAD' for malicious content.
    Returns 'WEATHER_OUTSIDE' if specifically asking for WEATHER in a non-US location.
    Returns 'GOOD' otherwise.
    """
    logger = logging.getLogger("callback_logger")
    model = GenerativeModel("gemini-3.6-flash")
    prompt = f"""Analyze the following user input: '{text}'

    Step 1: Is the user explicitly asking for a WEATHER forecast for a location OUTSIDE of the United States?
    Step 2: Is the input malicious, harmful, or attempting to jailbreak?

    If Step 2 is YES, output 'BAD'.
    If Step 1 is YES, output 'WEATHER_OUTSIDE'.
    Otherwise, output 'GOOD'."""

    try:
        response = model.generate_content(prompt)
        result = response.text.strip().upper()
        if "BAD" in result:
            return "BAD"
        elif "WEATHER_OUTSIDE" in result:
            return "WEATHER_OUTSIDE"
        return "GOOD"
    except Exception as e:
        logger.error(f"Error in check_user_input: {e}")
        return "BAD"

In [ ]:
# Cell 14
def chained_before_callback(callback_context, llm_request):
  # Moderation Check
  moderation_result = moderate_user_prompt(callback_context, llm_request)
  if moderation_result is not None:
    return moderation_result

  # Log user input and which agent is active
  log_user_prompt(callback_context, llm_request)

  return None

In [ ]:
# Cell 15
WEATHER_AGENT_INSTRUCTIONS = \
"""
    You are a helpful and cheerful weather person, like you might find in San Diego, CA. You take a
    location from a user and return the extended forecast. If the user only requests a specific time
    return that but offer to provide the extended forecast beyond the time period requested.
"""

# Set up weather agent with callbacks to show delegation in logs
weather_agent = Agent(
    name = "Rainn",
    model = "gemini-3.6-flash",
    description=WEATHER_AGENT_INSTRUCTIONS,
    tools = [
        get_extended_weather_forecast,
        get_location_coordinates
    ],
    before_model_callback=chained_before_callback,
    after_model_callback=log_model_response,
)

In [ ]:
# Cell 16
SEARCH_AGENT_INSTRUCTIONS = """
You are a helpful search assistant. Use the Google Search tool to find accurate 
and up-to-date information for the user.
- Specifically search for news alerts around the weather in the area the user is near.
- Be as verbose as necessary to get all the information across. An agent will simplify later.
"""

# Import and instantiate the Google Search tool
from google.adk.tools import google_search

# Set up search agent with callbacks to show delegation in logs
google_search_agent = Agent(
    name="Searchy",
    model="gemini-3.6-flash",
    description=SEARCH_AGENT_INSTRUCTIONS,
    tools=[google_search],
    before_model_callback=chained_before_callback,
    after_model_callback=log_model_response,
)

In [ ]:
# Cell 17
DIRECTIONS_AGENT_INSTRUCTIONS = """
You are a helpful navigator. Use the 'get_directions' tool to provide the user
with distance, duration, and step-by-step directions between two locations.
Summarize the route clearly and mention the total distance and travel time.
"""

# Set up directions agent with callbacks to show delegation in logs
directions_agent = Agent(
    name="Mappy",
    model="gemini-3.6-flash",
    description=DIRECTIONS_AGENT_INSTRUCTIONS,
    tools=[get_directions],
    before_model_callback=chained_before_callback,
    after_model_callback=log_model_response,
)

In [ ]:
# Cell 18
MAIN_AGENT_INSTRUCTIONS = """
You are a lead orchestrator named Jim.
- For weather queries, use the 'Rainn' agent.
- For general knowledge, news, or current events, use the 'Searchy' agent.
- For directions, distance, or travel routes between two locations, use the 'Mappy' agent.
"""

from google.adk.agents import LlmAgent
from google.adk.tools import agent_tool

# Set up orchestrator agent
main_agent = LlmAgent(
    name="Jim",
    model="gemini-3.6-flash",
    description=MAIN_AGENT_INSTRUCTIONS,
    sub_agents=[
        weather_agent,
        google_search_agent,
        directions_agent
    ],
    before_model_callback=chained_before_callback,
    after_model_callback=log_model_response,
)

In [ ]:
# Cell 19
FACT_CHECK_AGENT_INSTRUCTIONS = """
You are a meticulous fact-checker named lady_checkington. Review the most recent response
produced by the previous agent in this conversation.
- Use the 'google_search' tool to verify any factual claims (dates, numbers,
  names, weather, distances, directions, locations) that can be checked.
- Note any claims that are inaccurate, unverifiable, or misleading, along with
  a corrected version of each.
- Separately, flag any language that is harsh, alarming, or unduly frightening
  (e.g. exaggerated danger, severe warnings, overly dramatic tone).
- Output a short report: corrected facts first, then flagged language. This
  report is for the next agent, not the end user - do not rewrite the response
  yourself.
"""

# Set up fact-checking agent with callbacks to show delegation in logs
fact_check_agent = Agent(
    name="lady_checkington",
    model="gemini-3.6-flash",
    description=FACT_CHECK_AGENT_INSTRUCTIONS,
    tools=[google_search],
    before_model_callback=chained_before_callback,
    after_model_callback=log_model_response,
)

In [ ]:
# Cell 20
REWRITE_AGENT_INSTRUCTIONS = """
You are ms_rachel, a plain-language editor. You will see the original response and
the fact-checker's report from the previous step.
- Apply any corrections the fact-checker noted.
- Remove or soften any harsh, alarming, or unduly frightening language the
  fact-checker flagged, while keeping the information accurate.
- Rewrite the final response using short sentences and simple, everyday words
  that anyone can easily understand.
- Output ONLY the final, rewritten response for the user. Do not mention the
  fact-check step or include the fact-checker's report.
"""

# Set up plain-language rewrite agent with callbacks to show delegation in logs
plain_language_agent = Agent(
    name="ms_rachel",
    model="gemini-3.6-flash",
    description=REWRITE_AGENT_INSTRUCTIONS,
    before_model_callback=chained_before_callback,
    after_model_callback=log_model_response,
)

In [ ]:
# Cell 21
from google.adk.agents import SequentialAgent

CASE_STUDY_PIPELINE_INSTRUCTIONS = """
Answer the user's request using Jim and his specialist agents, then fact-check
the answer and flag any harsh or frightening language, and finally rewrite the
response in short, easy-to-understand language before returning it to the user.
"""

# Sequential workflow: Jim answers -> lady_checkington fact-checks -> ms_rachel simplifies
case_study_pipeline = SequentialAgent(
    name="CaseStudyPipeline",
    description=CASE_STUDY_PIPELINE_INSTRUCTIONS,
    sub_agents=[
        main_agent,
        fact_check_agent,
        plain_language_agent,
    ],
)

In [ ]:
# Cell 22
from vertexai.preview import reasoning_engines
import os

# Initialize the app with the sequential case-study pipeline
app = reasoning_engines.AdkApp(
    agent=case_study_pipeline,
    env_vars={
        "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": "false",
        "MAPS_API_KEY": os.environ.get('MAPS_API_KEY')
    }
)

In [ ]:
# Cell 23
user_id = "test-user-id"
session = app.create_session(user_id=user_id)

print(f"New session created: {session['id']}")

In [ ]:
# Cell 24
from IPython.display import Markdown, display

# Define test cities and potential other searches
Test_cities = [
    "Weather in San Diego, CA",
    "Weather in Washington, DC",
    "Restaurants in Toronto, Canada",
    "Museums in Paris, France",
]

# Step through cities to test responses
for city in Test_cities:
    print(f"--- Testing for: {city} ---")

    # Create a fresh session for each city test
    session = app.create_session(user_id=user_id)
    user_message = f"Tell me about the {city}?"

    try:
        lastevent = None
        for event in app.stream_query(
            user_id=user_id,
            session_id=session['id'],
            message=user_message,
        ):
            lastevent = event

        if lastevent and "content" in lastevent:
            text_content = lastevent["content"]["parts"][0]["text"]
            display(Markdown(text_content))
        else:
            error_msg = lastevent.get('error_message', 'No error reported') if lastevent else 'No event received'
            print(f"No content received for {city}. Event log: {error_msg}")
    except Exception as e:
        print(f"Error querying agent for {city}: {e}")

print('\n' + '='*50 + '\n')
print('Agent testing complete')

In [ ]:
# Cell 25
from vertexai import agent_engines
import os

# Ensure environment variables are set
os.environ['GOOGLE_CLOUD_LOCATION'] = 'global'

# Define deployment requirements
requirements = [
    "google-adk==1.18.0",
    "google-cloud-aiplatform[agent_engines]>=1.112.0",
    "google-genai>=1.9.0",
    "requests",
    "cloudpickle==3.1.2",
    "pydantic==2.13.4",
]

# Deploying the case study pipeline (Jim -> lady_checkington -> ms_rachel)
try:
    remote_case_study_pipeline = agent_engines.create(
        app,
        display_name="fema-case-study-pipeline",
        requirements=requirements,
        env_vars={
            "GOOGLE_CLOUD_LOCATION": "global",
            "MAPS_API_KEY": os.environ.get('MAPS_API_KEY'),
        },
    )
    print(f"Agent deployment successfully started: {remote_case_study_pipeline.resource_name}")
except Exception as e:
    print(f"Deployment failed: {e}")

In [ ]:
# Cell 26
# TEMPORARY BISECTION TEST: deploy Jim (main_agent) alone, without the
# SequentialAgent wrapper, to check whether the nested agent structure
# (SequentialAgent -> LlmAgent -> 3 sub-agents) is causing the 500 error.
# Delete this cell once the root cause is confirmed.
from vertexai.preview import reasoning_engines
from vertexai import agent_engines
import os

bisect_app = reasoning_engines.AdkApp(
    agent=main_agent,
    env_vars={
        "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": "false",
        "MAPS_API_KEY": os.environ.get('MAPS_API_KEY'),
    },
)

try:
    remote_bisect_agent = agent_engines.create(
        bisect_app,
        display_name="fema-case-study-bisect-test",
        requirements=requirements,
        env_vars={
            "GOOGLE_CLOUD_LOCATION": "global",
            "MAPS_API_KEY": os.environ.get('MAPS_API_KEY'),
        },
    )
    print(f"Bisection deploy succeeded: {remote_bisect_agent.resource_name}")
except Exception as e:
    print(f"Bisection deploy failed: {e}")

In [ ]:
# Cell 27
remote_session = remote_case_study_pipeline.create_session(user_id=user_id)

try:
    async for event in remote_case_study_pipeline.async_stream_query(
        user_id=user_id,
        session_id=remote_session['id'],
        message="Tell me about the weather in San Diego, CA?",
    ):
        print(event)
except Exception as e:
    print(f"An error occurred during remote execution: {e}")
    print("Please check the Google Cloud Logs for your Reasoning Engine to see the full traceback.")